# M3-B Research Runner — Colab Run All (synthetic acceptance + Drive archive)

This notebook runs one reviewed **research configuration**, `configs/research/sma_5_20_tsm_synthetic_v1.json`: the fixed SMA 5/20 strategy on the committed **synthetic fixture**, with 2 windows × 4 cost scenarios. It runs the configuration **twice**, then verifies the Google Drive archive.

Flow: Spec → Governance → Data → Signal → Core `ExecutionEngine.run` → Evaluation → Registry → Report → Archive.

* **Nothing local needed.** Open the notebook from GitHub in Colab and press Run All. No local git or Python is used; Python ≥ 3.11 is checked.
* **Pinned code.** The agent is cloned at the exact `AGENT_COMMIT`. Core is installed at the exact `CORE_SHA` (`BENCHMARK_PASS`). Only the repository's declared dependencies are installed.
* **No strategy, backtest or cost logic here.** Cells call `python -m strategy_agent.research` and check files. Parameters come only from the reviewed config; R02 is mandatory.
* **Credentials.** None are needed: both repositories are public. An optional `GITHUB_TOKEN` is read only from **Colab Secrets** and is never printed or persisted.
* **Isolation.** The run uses an isolated governance and registry under `/content/m3b_env`. The repository's `registry/` and `governance/` are fingerprinted before and after the run.
* **Real TSM stays blocked.** The market gate (`governance/market_readiness.json`) is **NO_GO**. The readiness cell must exit 3.
* **Archive location.** `MyDrive/Quant_Research/strategy-research-agent/YYYYMMDD_HHMMSS_<commit7>/` (Asia/Taipei).
* **Failure is loud.** Run All stops on the first failed check. Nothing is reported as backed up unless every manifest verifies on Drive.


In [ ]:
# --- configuration (pinned; change only together with a reviewed commit) ---
AGENT_REPO   = "https://github.com/hh4832/strategy-research-agent.git"
AGENT_COMMIT = "720b1744b0832ffe57684a925248e7e1e7b4d48a"                       # exact 40-hex commit, never a branch
CORE_SHA     = "2c6ba9ab7c336e7a4908744abef617e0dce713fd"   # BENCHMARK_PASS (M1 final)
WORKDIR      = "/content/strategy-research-agent"
RUN_ROOT     = "/content/m3b_env"                       # isolated synthetic governance + registry (not on Drive)
CONFIG       = "configs/research/sma_5_20_tsm_synthetic_v1.json"   # reviewed research configuration (DRIVE output)
DRIVE_ROOT   = "/content/drive/MyDrive/Quant_Research/strategy-research-agent"
import hashlib, json, os, pathlib, re, subprocess, sys
assert sys.version_info >= (3, 11), f"Python >= 3.11 required, found {sys.version}"
assert re.fullmatch(r"[0-9a-f]{40}", AGENT_COMMIT), "AGENT_COMMIT must be a full commit SHA"
assert re.fullmatch(r"[0-9a-f]{40}", CORE_SHA)


In [ ]:
# --- Google Drive (required: the archive target). Run All stops here if Drive cannot be mounted ---
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), "Google Drive is not mounted"


In [ ]:
# --- optional Colab Secret (not needed for public repos). Never printed or written to disk ---
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
print("GITHUB_TOKEN from Colab Secrets:", "present" if GITHUB_TOKEN else "not set (public clone)")


In [ ]:
# --- clone the agent at AGENT_COMMIT, install it and Core at CORE_SHA ---
def sh(cmd, secret_env=None, check=True):
    print("$", cmd)
    env = dict(os.environ, **(secret_env or {}))
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=env)
    out = (r.stdout[-6000:] + r.stderr[-3000:])
    if GITHUB_TOKEN:
        out = out.replace(GITHUB_TOKEN, "***")
    print(out)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r
sh(f"rm -rf {WORKDIR}")
if GITHUB_TOKEN:   # header passed via env for this one command only; not stored in .git/config
    sh(f'git -c http.extraHeader="Authorization: Bearer $GH_TOKEN" clone -q {AGENT_REPO} {WORKDIR}',
       secret_env={"GH_TOKEN": GITHUB_TOKEN})
else:
    sh(f"git clone -q {AGENT_REPO} {WORKDIR}")
sh(f"cd {WORKDIR} && git checkout -q {AGENT_COMMIT}")
head = sh(f"cd {WORKDIR} && git rev-parse HEAD").stdout.strip()
assert head == AGENT_COMMIT, (head, AGENT_COMMIT)
assert sh(f"cd {WORKDIR} && git status --porcelain").stdout.strip() == "", "checkout is not clean"
assert "extraheader" not in open(f"{WORKDIR}/.git/config").read().lower(), "credential persisted in .git/config"
sh(f"cd {WORKDIR} && pip install -q -e '.[test,research]' "
   f"'market-research-core @ git+https://github.com/hh4832/market-research-core@{CORE_SHA}'")


In [ ]:
# --- identity: agent commit, Core commit, BENCHMARK_PASS derived from the committed evidence record ---
r = sh(f"cd {WORKDIR} && python -c \"import json; from strategy_agent.adapters import mrc_adapter as A; "
       f"from strategy_agent.research.pipeline import agent_identity; "
       f"print(json.dumps({{'core': A.core_identity(), 'engine': A.engine_info().__dict__, 'agent': agent_identity()}}))\"")
ident = json.loads(r.stdout.strip().splitlines()[-1])
assert ident["core"]["git_sha"] == CORE_SHA and ident["engine"]["benchmark_status"] == "BENCHMARK_PASS"
assert ident["agent"]["agent_sha"] == AGENT_COMMIT and ident["agent"]["agent_tree_dirty"] is False


In [ ]:
# --- tests: M3-B unit + synthetic integration on the real Core, M1 Core compatibility ---
sh(f"cd {WORKDIR} && python -m pytest -q tests/test_m3b_unit.py tests/test_m3b_integration.py tests/test_m3b2_contract.py tests/test_m1_core_compat.py")


In [ ]:
# --- M3-B2 gate: Codex M3-A is NO-GO, so the real TSM run must refuse (exit 3) ---
r = sh(f"cd {WORKDIR} && python -m strategy_agent.research readiness --market IBKR_TSM", check=False)
assert r.returncode == 3, "IBKR_TSM gate is expected to be closed (M3-A NO-GO)"
r = sh(f"cd {WORKDIR} && python -m strategy_agent.research real --market IBKR_TSM", check=False)
assert r.returncode == 3 and "NOT STARTED" in r.stdout


In [ ]:
# --- formal registry / holdout fingerprint BEFORE the runs ---
def formal_digest():
    root = pathlib.Path(WORKDIR)
    files = sorted(list((root / "registry").glob("*")) + list((root / "governance").rglob("*")))
    return {str(p.relative_to(root)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files if p.is_file()}
FORMAL_BEFORE = formal_digest()
assert not pathlib.Path(RUN_ROOT).resolve().is_relative_to(pathlib.Path(WORKDIR).resolve())
print(len(FORMAL_BEFORE), "formal files fingerprinted")


In [ ]:
# --- run 1: the research configuration (2 windows x 4 cost scenarios), archived to Drive ---
r = sh(f"cd {WORKDIR} && python -m strategy_agent.research run --config {CONFIG} --run-root {RUN_ROOT} "
       f"--summary-out /content/m3b_run1.json", check=False)
assert r.returncode == 0, "run 1 failed or Drive archive unavailable (see output)"
S1 = json.load(open('/content/m3b_run1.json'))
assert S1["config"]["sha256"] and S1["config"]["kind"] == "SYNTHETIC"
RUNS1 = S1['runs']
assert RUNS1 and all(o['status'] == 'COMPLETED' and o['archive_status'] == 'ARCHIVED' for o in RUNS1)
DIRS1 = [pathlib.Path(o['archive_path']) for o in RUNS1]
assert all(d.is_relative_to(pathlib.Path(DRIVE_ROOT)) for d in DIRS1), DIRS1
SNAP1 = {str(d): {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in d.iterdir()} for d in DIRS1}
print(len(DIRS1), "runs archived:", [d.name for d in DIRS1])


In [ ]:
# --- run 2: the same configuration again (must reproduce, must not overwrite run 1) ---
r = sh(f"cd {WORKDIR} && python -m strategy_agent.research run --config {CONFIG} --run-root {RUN_ROOT} "
       f"--summary-out /content/m3b_run2.json", check=False)
assert r.returncode == 0, "run 2 failed or Drive archive unavailable (see output)"
S2 = json.load(open('/content/m3b_run2.json'))
RUNS2 = S2['runs']
DIRS2 = [pathlib.Path(o['archive_path']) for o in RUNS2]
print([d.name for d in DIRS2])


In [ ]:
# --- acceptance checks (Drive archive) ---
REQUIRED = ["run_info.txt", "experiment_record.json", "stages.json", "metrics.json", "research_report.md",
            "trades.csv", "fills.csv", "rejections.csv", "equity_curve.csv", "snapshot_refs.json",
            "final_status.json", "manifest.json", "research_config.json", "cost_scenario.json"]
checks = {}
def manifest_ok(d):
    man = json.loads((d / "manifest.json").read_text())["files"]
    return set(man) == {p.name for p in d.iterdir()} - {"manifest.json"} and \
        all(hashlib.sha256((d / f).read_bytes()).hexdigest() == v["sha256"] for f, v in man.items())
for tag, dirs in (("run1", DIRS1), ("run2", DIRS2)):
    checks[f"{tag}_required_files"] = all((d / f).is_file() for d in dirs for f in REQUIRED)
    checks[f"{tag}_manifest_every_file_sha256"] = all(manifest_ok(d) for d in dirs)
    checks[f"{tag}_engineering_pass"] = all(json.loads((d / "final_status.json").read_text())["status"] == "PASS" for d in dirs)
    checks[f"{tag}_on_drive"] = all(d.is_relative_to(pathlib.Path(DRIVE_ROOT)) for d in dirs)
checks["new_directory_per_run"] = len(set(DIRS1) | set(DIRS2)) == len(DIRS1) + len(DIRS2)
checks["run1_not_overwritten"] = all({p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in d.iterdir()} == SNAP1[str(d)]
                                     for d in DIRS1)
checks["reproducible_metrics"] = [o["metrics_sha256"] for o in RUNS1] == [o["metrics_sha256"] for o in RUNS2] and \
    [o["identity"] for o in RUNS1] == [o["identity"] for o in RUNS2]
checks["different_experiment_ids"] = not ({o["experiment_id"] for o in RUNS1} & {o["experiment_id"] for o in RUNS2})
checks["cost_scenarios_one_family_trial"] = S1["cost_sensitivity"]["counts_as_new_family_or_trial"] is False
checks["formal_registry_holdout_untouched"] = formal_digest() == FORMAL_BEFORE
checks["git_tree_clean"] = sh(f"cd {WORKDIR} && git status --porcelain").stdout.strip() == ""
checks["run_info_mentions_commit"] = all(AGENT_COMMIT in (d / "run_info.txt").read_text() for d in DIRS1)
print(json.dumps(checks, indent=1))
assert all(checks.values()), [k for k, v in checks.items() if not v]


In [ ]:
# --- write the acceptance record next to the archives (Drive) ---
from datetime import datetime
from zoneinfo import ZoneInfo
acc_dir = pathlib.Path(DRIVE_ROOT) / "_acceptance"
acc_dir.mkdir(parents=True, exist_ok=True)
acc = {"acceptance": "M3-B.2 Colab Run All + Drive archive", "result": "PASS", "agent_commit": AGENT_COMMIT,
       "core_sha": CORE_SHA, "config": S1["config"], "python": sys.version,
       "run1": [str(d) for d in DIRS1], "run2": [str(d) for d in DIRS2], "checks": checks,
       "at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat()}
p = acc_dir / f"m3b2_colab_{datetime.now(ZoneInfo('Asia/Taipei')):%Y%m%d_%H%M%S}_{AGENT_COMMIT[:7]}.json"
p.write_text(json.dumps(acc, indent=1, default=str))
print("M3-B.2 Colab Run All / Drive archive acceptance: PASS ->", p)


In [ ]:
# --- show the cost-sensitivity table and one archived report ---
from IPython.display import Markdown, display
display(Markdown(pathlib.Path(S1["summary_path"]).with_name(f"cost_sensitivity_{S1['batch_id']}.md").read_text()))
print((DIRS1[0] / "run_info.txt").read_text())
display(Markdown((DIRS1[0] / "research_report.md").read_text()))
